# Persiapan Dataset BananaLSD

Kode dari `prepare_data.py`, disusun menjadi sel notebook. Jalankan sel berurutan dengan kernel Python yang memiliki Pillow (`pip install Pillow`). Notebook mendukung direktori kerja root proyek atau folder `training`.

Sel terakhir menjalankan pemeriksaan gambar, deduplikasi, pembagian train/val/test, serta penyimpanan laporan. Saat dijalankan, folder `dataset/` dibuat ulang; gambar sumber di `OriginalSet/` tetap disimpan. Pengelompokan daun menggunakan kedekatan nomor file sebagai perkiraan, bukan identitas daun yang terverifikasi.

## Import dan konfigurasi

In [1]:
import hashlib
import json
import math
import os
import random
import shutil
import sys
from collections import defaultdict
from pathlib import Path

from PIL import Image

# -- konfigurasi ---------------------------------------------------------------

BASE_DIR = Path.cwd().resolve()
if not (BASE_DIR / "OriginalSet").is_dir():
    BASE_DIR = BASE_DIR / "training"
if not (BASE_DIR / "OriginalSet").is_dir():
    raise FileNotFoundError("Jalankan notebook dari root proyek atau folder training yang berisi OriginalSet.")

SEED = 42
ORIGINAL_DIR = BASE_DIR / "OriginalSet"
OUTPUT_DIR = BASE_DIR / "dataset"
SPLIT_MANIFEST = BASE_DIR / "artifacts" / "split_manifest.json"
QA_REPORT = BASE_DIR / "artifacts" / "data_qa_report.json"

CLASSES = ["cordana", "healthy", "pestalotiopsis", "sigatoka"]
VALID_EXTENSIONS = {".jpeg", ".jpg", ".png"}

TEST_RATIO = 0.20       # 20% data -> test
VAL_RATIO_OF_DEV = 0.25 # 25% dari development -> val  (sekitar 20% total)

# -- utilitas ------------------------------------------------------------------

## `pixel_hash`

In [2]:
def pixel_hash(path):
    """SHA-256 dari data piksel mentah (bukan file byte, jadi tahan metadata)."""
    try:
        img = Image.open(path)
        img.load()
        raw = img.tobytes()
        return hashlib.sha256(raw).hexdigest()
    except Exception:
        return None

## `check_integrity`

In [3]:
def check_integrity(root, classes):
    """Periksa gambar rusak, ekstensi salah, dan duplikat lintas/intra kelas."""
    corrupt = []
    bad_ext = []
    hash_to_paths = defaultdict(list)
    all_files_per_class = defaultdict(list)
    class_counts = {}

    for cls in classes:
        cls_dir = root / cls
        if not cls_dir.is_dir():
            print(f"  [PERINGATAN] Folder kelas tidak ditemukan: {cls_dir}")
            continue

        files = sorted(cls_dir.iterdir())
        class_counts[cls] = 0

        for fp in files:
            if not fp.is_file():
                continue
            rel = fp.relative_to(root).as_posix()

            if fp.suffix.lower() not in VALID_EXTENSIONS:
                bad_ext.append(rel)
                continue

            h = pixel_hash(fp)
            if h is None:
                corrupt.append(rel)
                continue

            hash_to_paths[h].append(rel)
            all_files_per_class[cls].append((rel, h))
            class_counts[cls] += 1

    duplicates = {h: paths for h, paths in hash_to_paths.items() if len(paths) > 1}

    # Deduplikasi: untuk setiap hash, simpan hanya file pertama (sorted)
    seen_hashes = set()
    valid_files = defaultdict(list)
    removed_duplicates = []

    for cls in classes:
        for rel, h in all_files_per_class.get(cls, []):
            if h in seen_hashes:
                removed_duplicates.append(rel)
                continue
            seen_hashes.add(h)
            valid_files[cls].append(rel)

    return {
        "class_counts_raw": class_counts,
        "corrupt": corrupt,
        "bad_extension": bad_ext,
        "duplicates": duplicates,
        "removed_duplicates": removed_duplicates,
        "valid_files": valid_files,
    }

## `get_file_number`

In [4]:
def get_file_number(rel_path):
    """Ambil nomor dari nama file."""
    stem = Path(rel_path).stem
    try:
        return int(stem)
    except ValueError:
        return None

## `build_leaf_groups`

In [5]:
def build_leaf_groups(file_list):
    """Kelompokkan file berdasarkan kedekatan nomor (proxy leaf-group).

    Dataset BananaLSD menamai gambar sebagai <nomor>.jpeg.
    Gambar dari daun yang sama biasanya memiliki nomor yang berdekatan
    (beberapa angle/crop dari satu daun). Kita kelompokkan file yang
    nomornya berselisih <= 1 menjadi satu grup.

    Agar tidak terbentuk satu grup raksasa, batasi ukuran
    maksimum grup menjadi 5 gambar.
    """
    MAX_GROUP_SIZE = 5
    MAX_GAP = 1

    numbered = []
    non_numbered = []

    for f in file_list:
        num = get_file_number(f)
        if num is not None:
            numbered.append((num, f))
        else:
            non_numbered.append(f)

    numbered.sort(key=lambda x: x[0])

    groups = []
    current_group = []
    prev_num = None

    for num, f in numbered:
        if prev_num is not None and (num - prev_num > MAX_GAP or len(current_group) >= MAX_GROUP_SIZE):
            groups.append(current_group)
            current_group = []
        current_group.append(f)
        prev_num = num

    if current_group:
        groups.append(current_group)

    for f in non_numbered:
        groups.append([f])

    return groups

## `split_groups_by_ratio`

In [6]:
def split_groups_by_ratio(groups, ratio, rng):
    """Split grup secara acak, mengembalikan (bagian_a, bagian_b).
    bagian_b mendapat sekitar `ratio` dari total file."""
    rng.shuffle(groups)
    total = sum(len(g) for g in groups)
    target_b = math.ceil(total * ratio)

    part_b = []
    part_a = []
    count_b = 0

    for g in groups:
        if count_b < target_b:
            part_b.extend(g)
            count_b += len(g)
        else:
            part_a.extend(g)

    return part_a, part_b

## `copy_split`

In [7]:
def copy_split(root, output_dir, split_name, cls, files):
    """Salin file ke folder output/<split>/<kelas>/."""
    dest = output_dir / split_name / cls
    dest.mkdir(parents=True, exist_ok=True)
    for rel in files:
        src = root / rel
        shutil.copy2(src, dest / src.name)

## Alur persiapan data

In [8]:
def main():
    print("=" * 64)
    print("  PERSIAPAN DATASET BananaLSD")
    print("=" * 64)

    if not ORIGINAL_DIR.is_dir():
        print(f"[ERROR] Folder OriginalSet tidak ditemukan: {ORIGINAL_DIR}")
        sys.exit(1)

    # -- langkah 1-3: pemeriksaan kualitas data ----------------------------
    print("\n[1/6] Memeriksa integritas gambar ...")
    qa = check_integrity(ORIGINAL_DIR, CLASSES)

    print(f"\n  Jumlah gambar per kelas (sebelum deduplikasi):")
    for cls in CLASSES:
        n = qa["class_counts_raw"].get(cls, 0)
        print(f"    {cls:20s}: {n}")
    print(f"    {'TOTAL':20s}: {sum(qa['class_counts_raw'].values())}")

    if qa["corrupt"]:
        print(f"\n  [!] Gambar rusak ({len(qa['corrupt'])}):")
        for f in qa["corrupt"]:
            print(f"    - {f}")
    else:
        print("\n  [OK] Tidak ada gambar rusak.")

    if qa["bad_extension"]:
        print(f"\n  [!] Ekstensi tidak valid ({len(qa['bad_extension'])}):")
        for f in qa["bad_extension"]:
            print(f"    - {f}")
    else:
        print("\n  [OK] Semua ekstensi valid.")

    n_dup_groups = len(qa["duplicates"])
    n_dup_files = sum(len(v) for v in qa["duplicates"].values())
    if qa["duplicates"]:
        print(f"\n  [!] Duplikat terdeteksi: {n_dup_groups} grup, {n_dup_files} file total")
        for h, paths in list(qa["duplicates"].items())[:5]:
            print(f"    hash ..{h[:12]}:")
            for p in paths:
                print(f"      - {p}")
        if n_dup_groups > 5:
            print(f"    ... dan {n_dup_groups - 5} grup lainnya.")
    else:
        print("\n  [OK] Tidak ada duplikat.")

    # -- langkah 2: deduplikasi --------------------------------------------
    print(f"\n[2/6] Deduplikasi ...")
    print(f"  Duplikat dihapus: {len(qa['removed_duplicates'])} file")

    total_valid = 0
    print(f"\n  Jumlah gambar per kelas (setelah deduplikasi):")
    for cls in CLASSES:
        n = len(qa["valid_files"].get(cls, []))
        total_valid += n
        print(f"    {cls:20s}: {n}")
    print(f"    {'TOTAL':20s}: {total_valid}")

    # -- langkah 3-4: pembagian data ---------------------------------------
    print("\n[3/6] Membangun leaf-group dan membagi data ...")
    rng = random.Random(SEED)

    manifest = {
        "seed": SEED,
        "ratios": {
            "test": TEST_RATIO,
            "val_of_dev": VAL_RATIO_OF_DEV,
        },
        "splits": {},
    }

    summary_rows = []

    for cls in CLASSES:
        files = qa["valid_files"].get(cls, [])
        if not files:
            continue

        # Langkah 1: bagi menjadi dev (80%) dan test (20%)
        groups = build_leaf_groups(files)
        dev_files, test_files = split_groups_by_ratio(groups, TEST_RATIO, rng)

        # Langkah 2: bagi dev menjadi train (75%) dan val (25%)
        dev_groups = build_leaf_groups(dev_files)
        train_files, val_files = split_groups_by_ratio(dev_groups, VAL_RATIO_OF_DEV, rng)

        manifest["splits"][cls] = {
            "train": sorted(train_files),
            "val": sorted(val_files),
            "test": sorted(test_files),
        }

        summary_rows.append({
            "class": cls,
            "total": len(files),
            "train": len(train_files),
            "val": len(val_files),
            "test": len(test_files),
        })

    # tabel ringkasan
    print(f"\n  {'Kelas':20s} {'Total':>6s} {'Train':>6s} {'Val':>6s} {'Test':>6s}")
    print("  " + "-" * 44)
    t_total = t_train = t_val = t_test = 0
    for r in summary_rows:
        print(
            f"  {r['class']:20s} {r['total']:6d} {r['train']:6d} "
            f"{r['val']:6d} {r['test']:6d}"
        )
        t_total += r["total"]
        t_train += r["train"]
        t_val += r["val"]
        t_test += r["test"]
    print("  " + "-" * 44)
    print(f"  {'TOTAL':20s} {t_total:6d} {t_train:6d} {t_val:6d} {t_test:6d}")
    if t_total > 0:
        print(
            f"  {'RASIO':20s} {'':>6s} "
            f"{t_train/t_total:.1%}  {t_val/t_total:.1%}  {t_test/t_total:.1%}"
        )

    # -- langkah 5: verifikasi tidak ada kebocoran antar split -------------
    print("\n[4/6] Memverifikasi tidak ada kebocoran antar split ...")
    leak_found = False
    for cls in CLASSES:
        sp = manifest["splits"].get(cls, {})
        sets = {name: set(sp.get(name, [])) for name in ["train", "val", "test"]}
        for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
            overlap = sets[a] & sets[b]
            if overlap:
                print(f"  [!] KEBOCORAN {cls}: {a} & {b} = {len(overlap)} file")
                leak_found = True
    if not leak_found:
        print("  [OK] Tidak ada kebocoran -- setiap gambar hanya ada di satu split.")

    # -- langkah 6: simpan manifest ----------------------------------------
    print("\n[5/6] Menyimpan manifest dan laporan QA ...")
    SPLIT_MANIFEST.parent.mkdir(parents=True, exist_ok=True)

    with open(SPLIT_MANIFEST, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)
    print(f"  [OK] Manifest disimpan: {SPLIT_MANIFEST}")

    qa_report = {
        "class_counts_raw": qa["class_counts_raw"],
        "total_valid_after_dedup": total_valid,
        "corrupt_files": qa["corrupt"],
        "bad_extension_files": qa["bad_extension"],
        "duplicate_groups": {h: paths for h, paths in qa["duplicates"].items()},
        "removed_duplicates": qa["removed_duplicates"],
        "split_summary": summary_rows,
    }
    with open(QA_REPORT, "w", encoding="utf-8") as f:
        json.dump(qa_report, f, indent=2, ensure_ascii=False)
    print(f"  [OK] Laporan QA disimpan: {QA_REPORT}")

    # -- langkah 7: salin file ke folder split -----------------------------
    print("\n[6/6] Menyalin file ke folder dataset/ ...")
    if OUTPUT_DIR.exists():
        shutil.rmtree(OUTPUT_DIR)

    for cls in CLASSES:
        sp = manifest["splits"].get(cls, {})
        for split_name in ["train", "val", "test"]:
            files = sp.get(split_name, [])
            copy_split(ORIGINAL_DIR, OUTPUT_DIR, split_name, cls, files)
            print(f"  [OK] {split_name}/{cls}: {len(files)} file disalin")
    print("\n" + "=" * 64)
    print("  PERSIAPAN DATA SELESAI")
    print("=" * 64)
    print(f"\n  Dataset siap di: {OUTPUT_DIR}")
    print(f"  Manifest split:  {SPLIT_MANIFEST}")
    print(f"  Laporan QA:      {QA_REPORT}")

## Jalankan persiapan data

Sel ini menulis laporan ke `artifacts/` dan mengganti isi folder `dataset/` dengan hasil pembagian terbaru.

In [9]:
main()

  PERSIAPAN DATASET BananaLSD

[1/6] Memeriksa integritas gambar ...


c:\PBL-KEL4_TI3C_SEM5\.venv\Lib\site-packages\PIL\TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
c:\PBL-KEL4_TI3C_SEM5\.venv\Lib\site-packages\PIL\JpegImagePlugin.py:874: UserWarning: Image appears to be a malformed MPO file, it will be interpreted as a base JPEG file
  warnings.warn(



  Jumlah gambar per kelas (sebelum deduplikasi):
    cordana             : 162
    healthy             : 129
    pestalotiopsis      : 173
    sigatoka            : 473
    TOTAL               : 937

  [OK] Tidak ada gambar rusak.

  [OK] Semua ekstensi valid.

  [!] Duplikat terdeteksi: 39 grup, 78 file total
    hash ..ece18024954d:
      - healthy/0.jpeg
      - healthy/104.jpeg
    hash ..e9d9a7a09fa6:
      - healthy/1.jpeg
      - healthy/109.jpeg
    hash ..378b21d4fb3c:
      - healthy/10.jpeg
      - healthy/106.jpeg
    hash ..2c8f97da3c3f:
      - healthy/101.jpeg
      - healthy/46.jpeg
    hash ..3b1c6d5d004a:
      - healthy/102.jpeg
      - healthy/39.jpeg
    ... dan 34 grup lainnya.

[2/6] Deduplikasi ...
  Duplikat dihapus: 39 file

  Jumlah gambar per kelas (setelah deduplikasi):
    cordana             : 162
    healthy             : 91
    pestalotiopsis      : 172
    sigatoka            : 473
    TOTAL               : 898

[3/6] Membangun leaf-group dan membagi 